# Boston crossing-density map

Self-contained pipeline: builds the ZIP study area, pulls crossings, erases water **and excluded areas (the airport)**, pulls walkable grass/green, clips everything, and writes a standalone interactive HTML map with the data baked in.

Edit **Config** for data/behaviour. Restyle via the **HTML template** cell (`:root` variables = theme; the `BASEMAPS` list = the in-map basemap dropdown). `BASEMAP_URL`/`BASEMAP_ATTR` in Config set the *default* basemap; the dropdown then switches among keyless presets live.

In [1]:
!pip install geopandas osmnx pygris shapely pandas pyogrio -q

## Config

In [2]:
TARGET_ZIPS = [
    "02108","02109","02110","02111","02113","02114","02115","02116",
    "02118","02119","02120","02121","02122","02124","02125","02127",
    "02128","02129","02130","02133","02134","02135","02138","02139",
    "02141","02142","02163","02199","02203","02205","02210","02215",
    "02445","02446","02458","02459","02467","02472",
]

# --- crossings ---
CROSSWALK_SOURCE = "osm"                     # "osm" nodes, or "file" for a downloaded GeoJSON
CROSSWALK_FILE   = "crosswalks.geojson"

# --- grass / green ---
GRASS_SOURCE    = "osm"                       # "osm" (fast, under-reports) or "massgis" (imagery, high recall)
MASSGIS_LC_FILE = "landcover_landuse_2016.gpkg"
GRASS_TAGS = {
    "landuse": ["grass","meadow","village_green","recreation_ground","cemetery","allotments"],
    "natural": ["grassland"],
    "leisure": ["park","garden","pitch","common","playground","dog_park","golf_course",
                "nature_reserve","recreation_ground"],
    "amenity": ["grave_yard"],
}
MIN_AREA_M2       = 150
DROP_PRIVATE      = True
DROP_HARD_SURFACE = True
DISSOLVE_GRASS    = False

# --- erase from study area + grass ---
ERASE_WATER  = True
WATER_TAGS   = {"natural": "water", "landuse": ["reservoir","basin"], "waterway": ["riverbank","dock"]}
EXCLUDE_TAGS = {"aeroway": "aerodrome"}      # airport airside grass isn't walkable; add "military" etc.

# --- appearance ---
# Default basemap (the in-map dropdown switches among keyless presets defined in the template cell).
# Injected as JSON, so quotes/links/apostrophes in BASEMAP_ATTR are safe.
BASEMAP_URL  = "https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png"
BASEMAP_ATTR = "&copy; OpenStreetMap contributors &copy; CARTO"

OUT_HTML = "boston_crosswalk_density.html"

## Imports

In [3]:
import json
import geopandas as gpd
import osmnx as ox
from shapely.ops import unary_union
from shapely.geometry import mapping

## Study area (ZIP boundary)
2020 cartographic-boundary ZCTAs via `pygris` (shoreline-clipped). Falls back to a local shapefile.

In [4]:
try:
    from pygris import zctas
    zcta_all = zctas(year=2020, cb=True, cache=True)
except Exception as e:
    print("pygris unavailable, reading local shapefile instead:", e)
    import pyogrio
    pyogrio.set_gdal_config_options({"SHAPE_RESTORE_SHX": "YES"})
    zcta_all = gpd.read_file("cb_2020_us_zcta520_500k.shp")

zip_col = next((c for c in ["ZCTA5CE20","ZCTA5CE10","GEOID20","GEOID10"] if c in zcta_all.columns), None)
assert zip_col, f"No ZIP column found. Columns present: {list(zcta_all.columns)}"

target = zcta_all[zcta_all[zip_col].isin(TARGET_ZIPS)].to_crs(4326)
assert len(target) > 0, "No ZCTAs matched TARGET_ZIPS"
print(f"matched {len(target)} of {len(TARGET_ZIPS)} target ZIPs on column {zip_col!r}")

study_area = unary_union(target.geometry)
BOUNDARY = {"type": "Feature", "properties": {}, "geometry": mapping(study_area)}

matched 38 of 38 target ZIPs on column 'ZCTA5CE20'


## Pedestrian crossings

In [5]:
if CROSSWALK_SOURCE == "osm":
    cw = ox.features_from_polygon(study_area, tags={"highway": "crossing"})
    cw = cw[cw.geometry.geom_type == "Point"]
    POINTS = [[float(g.y), float(g.x)] for g in cw.geometry]
else:
    cw = gpd.read_file(CROSSWALK_FILE).to_crs(4326)
    cw = gpd.clip(cw, study_area)
    reps = cw.geometry.representative_point()
    POINTS = [[float(g.y), float(g.x)] for g in reps]

print(len(POINTS), "crossing points")
assert POINTS, "No crossings found"

22451 crossing points


## Erase water & excluded areas (ponds, lakes, airport)
Subtracts OSM water and the excluded footprints (airport) from the study area, and keeps `erase_union` to cut them out of grass.

In [6]:
def _fetch_polys(tags):
    try:
        g = ox.features_from_polygon(study_area, tags=tags)
        g = g[g.geom_type.isin(["Polygon","MultiPolygon"])].copy()
        g["geometry"] = g.buffer(0)
        return g[~g.geometry.is_empty]
    except Exception as e:
        print("  fetch failed for", tags, ":", e)
        return None

erase_parts = []
if ERASE_WATER:
    w = _fetch_polys(WATER_TAGS)
    if w is not None and len(w): erase_parts.append(unary_union(w.geometry)); print(f"water: {len(w)} features")
if EXCLUDE_TAGS:
    x = _fetch_polys(EXCLUDE_TAGS)
    if x is not None and len(x): erase_parts.append(unary_union(x.geometry)); print(f"excluded (airport/etc.): {len(x)} features")

erase_union = unary_union(erase_parts) if erase_parts else None
if erase_union is not None:
    land_area = study_area.difference(erase_union)
    BOUNDARY = {"type": "Feature", "properties": {}, "geometry": mapping(land_area)}
    print("study area updated (water + exclusions removed)")
else:
    print("nothing erased; study area unchanged")

water: 163 features
excluded (airport/etc.): 1 features
study area updated (water + exclusions removed)


## Walkable grass / green space
OSM (broad tags + precision filters) or MassGIS land cover (grass class), then erase water/airport, drop slivers, clip. Watch the printed counts to see each filter's effect.

In [7]:
HARD = ["asphalt","concrete","paved","paving_stones","artificial_turf","tartan",
        "clay","sand","rubber","acrylic","dirt","gravel","fine_gravel","compacted"]

if GRASS_SOURCE == "massgis":
    lc = gpd.read_file(MASSGIS_LC_FILE)
    print("columns:", list(lc.columns))
    cls_col = next((c for c in lc.columns
                    if lc[c].dtype == object
                    and lc[c].astype(str).str.contains("grass", case=False, na=False).any()), None)
    assert cls_col, "No grass class column found; inspect columns above and set it manually"
    vals = sorted(lc.loc[lc[cls_col].astype(str).str.contains("grass", case=False, na=False), cls_col].unique())
    print(f"class column {cls_col!r}; grass values: {vals}")
    grass_gdf = lc[lc[cls_col].astype(str).str.contains("grass", case=False, na=False)].to_crs(4326)[["geometry"]]
else:
    grass_gdf = ox.features_from_polygon(study_area, tags=GRASS_TAGS)
    grass_gdf = grass_gdf[grass_gdf.geom_type.isin(["Polygon","MultiPolygon"])].copy()
    grass_gdf["geometry"] = grass_gdf.buffer(0)
    n0 = len(grass_gdf)
    if DROP_PRIVATE and "access" in grass_gdf.columns:
        grass_gdf = grass_gdf[~grass_gdf["access"].isin(["private","no","permit"])]
    if DROP_HARD_SURFACE and "surface" in grass_gdf.columns:
        grass_gdf = grass_gdf[~grass_gdf["surface"].astype(str).str.lower().isin(HARD)]
    print(f"OSM grass: {n0} raw -> {len(grass_gdf)} after access/surface filters")
    grass_gdf = grass_gdf[["geometry"]]

grass_gdf = grass_gdf[~grass_gdf.geometry.is_empty]
grass_gdf = gpd.clip(grass_gdf, study_area)

if erase_union is not None:
    grass_gdf["geometry"] = grass_gdf.geometry.difference(erase_union)
    grass_gdf = grass_gdf[~grass_gdf.geometry.is_empty]

grass_gdf = grass_gdf.explode(index_parts=False)
grass_gdf = grass_gdf[grass_gdf.geom_type.isin(["Polygon","MultiPolygon"])]

before = len(grass_gdf)
grass_gdf = grass_gdf[grass_gdf.to_crs(26986).area.values >= MIN_AREA_M2]
print(f"dropped {before - len(grass_gdf)} polygons under {MIN_AREA_M2} m\u00b2")

if DISSOLVE_GRASS and len(grass_gdf):
    grass_gdf = gpd.GeoDataFrame(geometry=[unary_union(grass_gdf.geometry)], crs=grass_gdf.crs).explode(index_parts=False)

area_km2 = grass_gdf.to_crs(26986).area.sum() / 1e6 if len(grass_gdf) else 0.0
print(f"{len(grass_gdf)} grass polygons, {area_km2:.2f} km\u00b2 total")
GRASS = json.loads(grass_gdf[["geometry"]].reset_index(drop=True).to_json()) if len(grass_gdf) \
        else {"type": "FeatureCollection", "features": []}

OSM grass: 5388 raw -> 5034 after access/surface filters
dropped 1371 polygons under 150 m²
3683 grass polygons, 30.22 km² total


## HTML template
The full page. **To restyle:** the `:root { ... }` block (marked `>>> UI THEME`) holds colors, accent hue, fonts, and panel styling; the **`BASEMAPS`** list (marked `>>> BASEMAPS`) defines the in-map basemap dropdown — all keyless presets, edit or extend freely. `%%...%%` markers are filled by the next cell.

In [8]:
TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Boston crossing density</title>
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.css">
<link rel="preconnect" href="https://fonts.googleapis.com"><link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Archivo:wght@500;700;900&family=IBM+Plex+Mono:wght@400;500;600&family=IBM+Plex+Sans:wght@400;500;600&display=swap" rel="stylesheet">
<style>
  /* >>> UI THEME: colors, accent hue, fonts, panel look all live in these variables */
  :root{--bg:#0c0f13;--panel:#12161c;--panel-2:#191f27;--line:#28313c;--line-soft:#1e2630;
    --ink:#e9eef4;--ink-dim:#93a1b0;--ink-faint:#5f6d7c;--hue:184;--accent:hsl(var(--hue),85%,52%);--accent-soft:hsl(var(--hue),40%,72%);--green:#41b06a;
    --radius:14px;--mono:'IBM Plex Mono',ui-monospace,Menlo,monospace;--body:'IBM Plex Sans',system-ui,sans-serif;--display:'Archivo',system-ui,sans-serif;}
  *{box-sizing:border-box}html,body{height:100%;margin:0}
  body{font-family:var(--body);background:var(--bg);color:var(--ink);overflow:hidden}
  #map{position:absolute;inset:0;background:var(--bg);z-index:0}
  .panel{position:absolute;top:16px;left:16px;z-index:1000;width:322px;max-height:calc(100% - 32px);display:flex;flex-direction:column;
    background:linear-gradient(180deg,var(--panel),#0f1319);border:1px solid var(--line);border-radius:var(--radius);box-shadow:0 18px 50px rgba(0,0,0,.55);overflow:hidden}
  .panel__scroll{overflow-y:auto;padding:0 18px 18px}
  .panel__scroll::-webkit-scrollbar{width:10px}.panel__scroll::-webkit-scrollbar-thumb{background:var(--line);border-radius:8px;border:3px solid var(--panel)}
  .head{padding:18px 18px 15px;border-bottom:1px solid var(--line-soft)}
  .zebra{height:9px;border-radius:2px;margin-bottom:13px;background:repeating-linear-gradient(90deg,var(--ink) 0 7px,transparent 7px 13px);opacity:.9}
  .eyebrow{font-family:var(--mono);font-size:10.5px;letter-spacing:.22em;text-transform:uppercase;color:var(--ink-faint);margin:0 0 4px}
  h1{font-family:var(--display);font-weight:900;font-size:25px;line-height:1.02;letter-spacing:-.02em;margin:0}
  h1 .u{color:var(--accent)}
  .stat{margin-top:9px;font-family:var(--mono);font-size:11px;color:var(--ink-dim)}
  .sec{padding-top:17px}.sec + .sec{border-top:1px solid var(--line-soft);margin-top:3px}
  .sec__label{font-family:var(--mono);font-size:10.5px;letter-spacing:.18em;text-transform:uppercase;color:var(--ink-dim);margin:0 0 11px;display:flex;justify-content:space-between}
  .seg{display:flex;background:#0c1015;border:1px solid var(--line);border-radius:9px;padding:3px;gap:3px}
  .seg button{flex:1;font-family:var(--body);font-size:13px;font-weight:600;color:var(--ink-dim);background:transparent;border:0;border-radius:6px;padding:8px 6px;cursor:pointer}
  .seg button.active{background:var(--accent);color:#06181b}
  .seg button:focus-visible{outline:2px solid var(--accent);outline-offset:2px}
  .sel{width:100%;font-family:var(--body);font-size:13px;color:var(--ink);background:#0c1015;border:1px solid var(--line);border-radius:9px;padding:9px 10px;cursor:pointer}
  .sel option{background:var(--panel);color:var(--ink)}
  .sel:focus-visible{outline:2px solid var(--accent);outline-offset:1px}
  .toggle-row{display:flex;align-items:center;gap:10px;font-size:13px}
  .toggle-row .sw{flex:0 0 auto;width:42px;height:24px;border-radius:12px;background:var(--line);position:relative;cursor:pointer;transition:background .15s}
  .toggle-row .sw::after{content:'';position:absolute;top:3px;left:3px;width:18px;height:18px;border-radius:50%;background:#fff;transition:left .15s}
  .toggle-row .sw.on{background:var(--green)}.toggle-row .sw.on::after{left:21px}
  .scale{height:16px;border-radius:4px;border:1px solid var(--line);background:linear-gradient(90deg,hsl(var(--hue),35%,78%),hsl(var(--hue),55%,66%),hsl(var(--hue),72%,55%),hsl(var(--hue),88%,47%),hsl(var(--hue),96%,42%))}
  .scale__ends{display:flex;justify-content:space-between;font-family:var(--mono);font-size:10.5px;color:var(--ink-dim);margin-top:6px}
  .grp[hidden]{display:none}.ctrl{margin-bottom:13px}.ctrl:last-child{margin-bottom:0}
  .ctrl__top{display:flex;justify-content:space-between;font-size:12.5px;color:var(--ink);margin-bottom:6px}
  .ctrl__top .v{font-family:var(--mono);color:var(--accent)}
  input[type=range]{width:100%;height:4px;-webkit-appearance:none;appearance:none;background:var(--line);border-radius:4px;outline:none}
  input[type=range]::-webkit-slider-thumb{-webkit-appearance:none;width:16px;height:16px;border-radius:50%;background:var(--accent);border:2px solid var(--panel);cursor:pointer}
  input[type=range]::-moz-range-thumb{width:16px;height:16px;border-radius:50%;background:var(--accent);border:2px solid var(--panel);cursor:pointer}
  .note{font-size:11.5px;line-height:1.5;color:var(--ink-dim)}.note[hidden]{display:none}.note b{color:var(--ink);font-weight:600}
  .toggle{position:absolute;top:16px;left:16px;z-index:1001;display:none;width:44px;height:44px;border-radius:11px;border:1px solid var(--line);background:var(--panel);color:var(--ink);cursor:pointer;font-size:18px}
  @media (max-width:640px){.panel{width:calc(100% - 32px);max-height:calc(100% - 80px);top:70px}.panel.collapsed{display:none}.toggle{display:block}}
  @media (prefers-reduced-motion:reduce){*{transition:none!important}}
  .leaflet-container{background:var(--bg)}
  .leaflet-control-attribution{background:rgba(12,15,19,.8)!important;color:var(--ink-faint)!important}
  .leaflet-control-attribution a{color:var(--ink-dim)!important}
  .leaflet-tooltip{background:var(--panel);color:var(--ink);border:1px solid var(--line);font-family:var(--mono);font-size:12px;padding:4px 8px}
  .leaflet-tooltip-top:before{border-top-color:var(--line)}
</style>
</head>
<body>
<div id="map"></div>
<button class="toggle" id="toggle" aria-label="Show or hide controls">☰</button>
<aside class="panel" id="panel">
  <div class="head">
    <div class="zebra" aria-hidden="true"></div>
    <p class="eyebrow">Boston · pedestrian crossings</p>
    <h1>Crossing <span class="u">density</span></h1>
    <div class="stat" id="stat">—</div>
  </div>
  <div class="panel__scroll">
    <div class="sec">
      <p class="sec__label">View mode</p>
      <div class="seg" role="group" aria-label="View mode">
        <button id="modeSurface" class="active" aria-pressed="true">Density</button>
        <button id="modeHex" aria-pressed="false">Hexbins</button>
      </div>
    </div>
    <div class="sec">
      <p class="sec__label">Basemap</p>
      <select id="basemapSel" class="sel" aria-label="Basemap"></select>
    </div>
    <div class="sec" id="greenSec">
      <p class="sec__label">Layers</p>
      <div class="toggle-row"><span class="sw on" id="greenSw" role="switch" aria-checked="true" tabindex="0"></span><span>Walkable grass / green</span></div>
    </div>
    <div class="sec">
      <p class="sec__label">Saturation = crossing density</p>
      <div class="scale" id="scale" aria-hidden="true"></div>
      <div class="scale__ends"><span id="endLo">0</span><span id="endHi">/km²</span></div>
    </div>
    <div class="sec">
      <p class="sec__label">Appearance</p>
      <div class="ctrl"><div class="ctrl__top"><span>Hue</span><span class="v" id="hueV">184°</span></div><input type="range" id="hue" min="0" max="360" value="184"></div>
      <div class="grp" id="grpSurface">
        <div class="ctrl"><div class="ctrl__top"><span>Bandwidth</span><span class="v" id="bwV">300 m</span></div><input type="range" id="bw" min="80" max="1000" step="10" value="300"></div>
        <div class="ctrl"><div class="ctrl__top"><span>Detail</span><span class="v" id="detailV">320 cells</span></div><input type="range" id="detail" min="120" max="500" step="10" value="320"></div>
        <div class="ctrl"><div class="ctrl__top"><span>Opacity</span><span class="v" id="opacityV">70%</span></div><input type="range" id="opacity" min="20" max="100" step="5" value="70"></div>
        <div class="ctrl"><div class="ctrl__top"><span>Max density</span><span class="v" id="maxDV">/km²</span></div><input type="range" id="maxD" min="5" max="1000" step="5" value="150"></div>
        <div class="ctrl"><div class="ctrl__top"><span>Fill</span></div>
          <div class="seg" role="group" aria-label="Fill style"><button id="fillSmooth" class="active" aria-pressed="true">Smooth</button><button id="fillStepped" aria-pressed="false">Stepped</button></div></div>
      </div>
      <div class="grp" id="grpHex" hidden>
        <div class="ctrl"><div class="ctrl__top"><span>Hex size</span><span class="v" id="hexSizeV">300 m</span></div><input type="range" id="hexSize" min="60" max="800" step="10" value="300"></div>
        <div class="ctrl"><div class="ctrl__top"><span>Count ceiling</span><span class="v" id="hexCeilV">12 / hex</span></div><input type="range" id="hexCeil" min="1" max="60" value="12"></div>
      </div>
    </div>
    <div class="sec">
      <p class="sec__label">Reading this map</p>
      <p class="note" id="noteSurface"><b>Kernel density estimate</b> (quartic kernel) — crossings per km² on a fixed scale. <b>Bandwidth</b> sets how localized the density is (lower = sharper peaks); <b>Detail</b> sharpens the rendering only. The surface is masked to the study-area boundary, so nothing shows over water or outside the ZIPs.</p>
      <p class="note" id="noteHex" hidden><b>Hover a cell for its count.</b> One value per fixed-geometry cell. Cells are kept when their center is inside the boundary. For an index, rebuild the tessellation in ArcGIS with zero-count cells and join.</p>
    </div>
  </div>
</aside>
<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>
<script>
(function(){
  "use strict";
  var $=function(id){return document.getElementById(id);};
  // ---- injected data ----
  var POINTS = %%POINTS%%;        // [[lat,lng], ...]
  var BOUNDARY = %%BOUNDARY%%;    // study-area Feature/geometry (already shoreline-clipped)
  var GRASS = %%GRASS%%;          // FeatureCollection, pre-clipped to the study area in Python

  var map=L.map('map',{zoomControl:false,minZoom:10,maxZoom:18}).setView([42.3601,-71.0589],12);
  L.control.zoom({position:'bottomright'}).addTo(map);
  map.createPane('greenPane'); map.getPane('greenPane').style.zIndex=350;
  // Config default (injected as JSON, so quotes/links in the attribution are safe)
  var CFG_URL  = %%BASEMAP_URL%%;
  var CFG_ATTR = %%BASEMAP_ATTR%%;
  var A_CARTO='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> &copy; <a href="https://carto.com/attributions">CARTO</a>';
  var A_OSM='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors';
  var A_ESRI='Tiles &copy; Esri &mdash; Esri, HERE, Garmin, &copy; OpenStreetMap contributors';
  var A_ESRI_IMG='Tiles &copy; Esri &mdash; Source: Esri, Maxar, Earthstar Geographics, and the GIS User Community';
  var A_OTM='&copy; <a href="https://opentopomap.org">OpenTopoMap</a> (CC-BY-SA)';
  // >>> BASEMAPS: all keyless. Edit/extend this list to change the dropdown. maxNativeZoom = each
  // provider's real max (Leaflet upscales beyond it rather than showing blank tiles).
  var BASEMAPS=[
    {name:'Config default',   url:CFG_URL, attr:CFG_ATTR, sub:'abcd', max:20},
    {name:'CARTO Dark',       url:'https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',  attr:A_CARTO, sub:'abcd', max:20},
    {name:'CARTO Positron',   url:'https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png', attr:A_CARTO, sub:'abcd', max:20},
    {name:'CARTO Voyager',    url:'https://{s}.basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}{r}.png', attr:A_CARTO, sub:'abcd', max:20},
    {name:'Esri Imagery',     url:'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', attr:A_ESRI_IMG, sub:'', max:19},
    {name:'Esri Light Gray',  url:'https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}', attr:A_ESRI, sub:'', max:16},
    {name:'Esri Dark Gray',   url:'https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}', attr:A_ESRI, sub:'', max:16},
    {name:'OpenStreetMap',    url:'https://tile.openstreetmap.org/{z}/{x}/{y}.png', attr:A_OSM, sub:'abc', max:19},
    {name:'OpenTopoMap',      url:'https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png', attr:A_OTM, sub:'abc', max:17}
  ];
  var baseLayer=null;
  function setBasemap(b){ if(baseLayer) map.removeLayer(baseLayer);
    baseLayer=L.tileLayer(b.url,{attribution:b.attr, subdomains:b.sub||'abc', maxNativeZoom:b.max||19, maxZoom:19}).addTo(map);
    baseLayer.bringToBack(); }
  (function(){ var sel=$('basemapSel');
    BASEMAPS.forEach(function(b,i){ var o=document.createElement('option'); o.value=String(i); o.textContent=b.name; sel.appendChild(o); });
    sel.value='0'; setBasemap(BASEMAPS[0]);
    sel.addEventListener('change',function(){ setBasemap(BASEMAPS[+sel.value]); }); })();

  var points=POINTS||[], mode='surface';
  var hexLayer=L.layerGroup(), surfaceOverlay=null, lastGrid=null;
  var clipRings=null, clipLayer=null, greenLayer=null, greenVisible=true;
  var opts={hue:184,bandwidth:300,detail:320,opacity:0.70,maxDensity:150,fill:'smooth',hexSize:300,hexCeil:12,autoFit:true};

  var LAT0=42.32,LNG0=-71.09,M_LAT=111320,M_LNG=111320*Math.cos(LAT0*Math.PI/180);
  function proj(lat,lng){return [(lng-LNG0)*M_LNG,(lat-LAT0)*M_LAT];}
  function unproj(x,y){return [LAT0+y/M_LAT,LNG0+x/M_LNG];}

  var STOPS=[[0,35,78],[0.35,55,66],[0.55,72,55],[0.78,88,47],[1,96,42]];
  function rampSL(t){t=Math.max(0,Math.min(1,t));for(var i=1;i<STOPS.length;i++){if(t<=STOPS[i][0]){var a=STOPS[i-1],b=STOPS[i],f=(t-a[0])/((b[0]-a[0])||1);return [a[1]+(b[1]-a[1])*f,a[2]+(b[2]-a[2])*f];}}return [96,42];}
  function rampColor(t,h){var sl=rampSL(t);return 'hsl('+h+','+sl[0].toFixed(1)+'%,'+sl[1].toFixed(1)+'%)';}
  function hslToRgb(h,s,l){s/=100;l/=100;var c=(1-Math.abs(2*l-1))*s,x=c*(1-Math.abs((h/60)%2-1)),m=l-c/2,r,g,b;
    if(h<60){r=c;g=x;b=0;}else if(h<120){r=x;g=c;b=0;}else if(h<180){r=0;g=c;b=x;}else if(h<240){r=0;g=x;b=c;}else if(h<300){r=x;g=0;b=c;}else{r=c;g=0;b=x;}
    return [Math.round((r+m)*255),Math.round((g+m)*255),Math.round((b+m)*255)];}
  function rampRGB(t,h){var sl=rampSL(t);return hslToRgb(h,sl[0],sl[1]);}

  function geomToRings(geom,out){if(!geom)return;if(geom.type==='Polygon')geom.coordinates.forEach(function(r){out.push(r);});
    else if(geom.type==='MultiPolygon')geom.coordinates.forEach(function(p){p.forEach(function(r){out.push(r);});});}
  function collectRings(gj){var out=[];var feats=gj.type==='FeatureCollection'?gj.features:(gj.type==='Feature'?[gj]:[{geometry:gj}]);
    feats.forEach(function(f){geomToRings(f.geometry||f,out);});return out;}
  function insideClip(lng,lat){if(!clipRings)return true;var inside=false;for(var r=0;r<clipRings.length;r++){var ring=clipRings[r];
    for(var i=0,j=ring.length-1;i<ring.length;j=i++){var yi=ring[i][1],yj=ring[j][1];if((yi>lat)!==(yj>lat)){var xi=ring[i][0],xj=ring[j][0];if(lng<(xj-xi)*(lat-yi)/(yj-yi)+xi)inside=!inside;}}}return inside;}
  function buildMask(nx,ny,minx,miny,cx,cy){if(!clipRings)return null;var mask=new Uint8Array(nx*ny);
    for(var jj=0;jj<ny;jj++){var lat=LAT0+(miny+(jj+0.5)*cy)/M_LAT,xs=[];
      for(var r=0;r<clipRings.length;r++){var ring=clipRings[r];for(var i=0,j=ring.length-1;i<ring.length;j=i++){var yi=ring[i][1],yj=ring[j][1];
        if((yi>lat)!==(yj>lat)){var xi=ring[i][0],xj=ring[j][0];xs.push((xj-xi)*(lat-yi)/(yj-yi)+xi);}}}
      if(!xs.length)continue;xs.sort(function(a,b){return a-b;});
      for(var ii=0;ii<nx;ii++){var lng=LNG0+(minx+(ii+0.5)*cx)/M_LNG,lo=0,hi=xs.length;while(lo<hi){var mid=(lo+hi)>>1;if(xs[mid]<lng)lo=mid+1;else hi=mid;}if(lo&1)mask[jj*nx+ii]=1;}}
    return mask;}

  function render(){mode==='surface'?renderSurface():renderHex();if(clipLayer)clipLayer.bringToFront();}

  function computeGrid(){if(!points.length){lastGrid=null;return;}
    var h=opts.bandwidth,h2=h*h,P=[],minx=Infinity,miny=Infinity,maxx=-Infinity,maxy=-Infinity;
    for(var i=0;i<points.length;i++){var m=proj(points[i][0],points[i][1]);P.push(m);if(m[0]<minx)minx=m[0];if(m[0]>maxx)maxx=m[0];if(m[1]<miny)miny=m[1];if(m[1]>maxy)maxy=m[1];}
    minx-=h;miny-=h;maxx+=h;maxy+=h;var W=maxx-minx,H=maxy-miny,longer=Math.max(W,H);
    var cell=Math.max(longer/opts.detail,12);var nx=Math.max(2,Math.min(Math.round(W/cell),520)),ny=Math.max(2,Math.min(Math.round(H/cell),520));
    var cx=W/nx,cy=H/ny,grid=new Float64Array(nx*ny),coef=3/(Math.PI*h2),rx=Math.ceil(h/cx),ry=Math.ceil(h/cy);
    for(var p=0;p<P.length;p++){var px=P[p][0],py=P[p][1],gi=(px-minx)/cx,gj=(py-miny)/cy;
      var i0=Math.max(0,Math.floor(gi)-rx),i1=Math.min(nx-1,Math.floor(gi)+rx),j0=Math.max(0,Math.floor(gj)-ry),j1=Math.min(ny-1,Math.floor(gj)+ry);
      for(var jj=j0;jj<=j1;jj++){var ccy=miny+(jj+0.5)*cy,ddy=ccy-py;for(var ii=i0;ii<=i1;ii++){var ccx=minx+(ii+0.5)*cx,ddx=ccx-px,d2=ddx*ddx+ddy*ddy;if(d2<h2){var w=1-d2/h2;grid[jj*nx+ii]+=coef*w*w;}}}}
    var peak=0;for(var k=0;k<grid.length;k++){if(grid[k]>peak)peak=grid[k];}
    lastGrid={grid:grid,nx:nx,ny:ny,minx:minx,miny:miny,cx:cx,cy:cy,maxx:maxx,maxy:maxy,peak:peak*1e6,mask:buildMask(nx,ny,minx,miny,cx,cy)};}
  function niceCeil(x){if(!(x>0))return 50;var e=Math.pow(10,Math.floor(Math.log10(x))),f=x/e,n=f<=1?1:f<=2?2:f<=5?5:10;return Math.round(n*e);}
  function maybeAutoFit(){if(opts.autoFit&&lastGrid){opts.maxDensity=Math.max(5,Math.min(1000,niceCeil(lastGrid.peak)));$('maxD').value=opts.maxDensity;$('maxDV').textContent=opts.maxDensity+' /km\u00b2';updateEnds();opts.autoFit=false;}}
  function paintSurface(){if(surfaceOverlay){map.removeLayer(surfaceOverlay);surfaceOverlay=null;}if(!lastGrid){setStat();return;}
    var g=lastGrid,nx=g.nx,ny=g.ny,maxD=opts.maxDensity,h=opts.hue,bands=6,stepped=(opts.fill==='stepped'),mask=g.mask,op=opts.opacity;
    var cvs=document.createElement('canvas');cvs.width=nx;cvs.height=ny;var ctx=cvs.getContext('2d'),img=ctx.createImageData(nx,ny),data=img.data;
    for(var jj=0;jj<ny;jj++){var imgRow=ny-1-jj;for(var ii=0;ii<nx;ii++){var gidx=jj*nx+ii,o=(imgRow*nx+ii)*4;
      if(mask&&!mask[gidx]){data[o+3]=0;continue;}var dens=g.grid[gidx]*1e6,t=maxD>0?dens/maxD:0;if(t>1)t=1;if(t<0)t=0;if(t<=0.02){data[o+3]=0;continue;}
      var tc=t,a;if(stepped){var idx=Math.min(bands-1,Math.floor(t*bands));tc=(idx+0.5)/bands;a=0.14+(idx/(bands-1))*0.72;}else{a=0.12+t*0.76;}
      var alpha=a*op;if(alpha>op)alpha=op;var rgb=rampRGB(tc,h);data[o]=rgb[0];data[o+1]=rgb[1];data[o+2]=rgb[2];data[o+3]=Math.round(alpha*255);}}
    ctx.putImageData(img,0,0);var sw=unproj(g.minx,g.miny),ne=unproj(g.maxx,g.maxy);
    surfaceOverlay=L.imageOverlay(cvs.toDataURL(),[[sw[0],sw[1]],[ne[0],ne[1]]],{opacity:1,interactive:false}).addTo(map);
    setStat(Math.round(g.peak)+'/km\u00b2 peak');}
  function renderSurface(){if(map.hasLayer(hexLayer))map.removeLayer(hexLayer);hexLayer.clearLayers();computeGrid();maybeAutoFit();paintSurface();}

  function cubeRound(q,r){var x=q,z=r,y=-x-z,rx=Math.round(x),ry=Math.round(y),rz=Math.round(z);var dx=Math.abs(rx-x),dy=Math.abs(ry-y),dz=Math.abs(rz-z);
    if(dx>dy&&dx>dz)rx=-ry-rz;else if(dy>dz)ry=-rx-rz;else rz=-rx-ry;return rx+','+rz;}
  function renderHex(){if(surfaceOverlay){map.removeLayer(surfaceOverlay);surfaceOverlay=null;}hexLayer.clearLayers();
    if(points.length){var R=opts.hexSize,bins={};
      for(var i=0;i<points.length;i++){var m=proj(points[i][0],points[i][1]);var q=(Math.sqrt(3)/3*m[0]-1/3*m[1])/R,r=(2/3*m[1])/R;var key=cubeRound(q,r);bins[key]=(bins[key]||0)+1;}
      var ceil=opts.hexCeil,cells=0,maxCount=0;
      for(var key in bins){var qr=key.split(','),qq=+qr[0],rr=+qr[1],cxm=R*Math.sqrt(3)*(qq+rr/2),cym=R*1.5*rr,c=unproj(cxm,cym);
        if(clipRings&&!insideClip(c[1],c[0]))continue;var count=bins[key];cells++;if(count>maxCount)maxCount=count;var verts=[];
        for(var v=0;v<6;v++){var ang=Math.PI/180*(60*v-30);verts.push(unproj(cxm+R*Math.cos(ang),cym+R*Math.sin(ang)));}
        L.polygon(verts,{stroke:true,color:'#0c0f13',weight:0.6,fillColor:rampColor(count/ceil,opts.hue),fillOpacity:0.82}).bindTooltip(count+' crossing'+(count===1?'':'s'),{sticky:true,direction:'top'}).addTo(hexLayer);}
      setStat(cells+' hexes, peak '+maxCount+'/cell');}else{setStat();}
    hexLayer.addTo(map);}

  function setStat(extra){var n=points.length.toLocaleString();var clip=clipRings?' \u00b7 clipped to study area':'';$('stat').textContent=n+' crossings'+clip+(extra?(' \u00b7 '+extra):'');}
  function updateEnds(){if(mode==='surface'){$('endLo').textContent='0';$('endHi').textContent=opts.maxDensity+' /km\u00b2';}else{$('endLo').textContent='1';$('endHi').textContent=opts.hexCeil+'+';}}

  // ---- controls ----
  var raf=null;function schedule(fn){if(raf)cancelAnimationFrame(raf);raf=requestAnimationFrame(function(){raf=null;fn();});}
  function bf(){if(clipLayer)clipLayer.bringToFront();}
  $('hue').addEventListener('input',function(){opts.hue=+this.value;document.documentElement.style.setProperty('--hue',opts.hue);$('hueV').textContent=opts.hue+'\u00b0';if(mode==='surface')paintSurface();else renderHex();bf();});
  $('bw').addEventListener('input',function(){opts.bandwidth=+this.value;$('bwV').textContent=this.value+' m';schedule(function(){computeGrid();paintSurface();bf();});});
  $('detail').addEventListener('input',function(){opts.detail=+this.value;$('detailV').textContent=this.value+' cells';schedule(function(){computeGrid();paintSurface();bf();});});
  $('opacity').addEventListener('input',function(){opts.opacity=(+this.value)/100;$('opacityV').textContent=this.value+'%';paintSurface();bf();});
  $('maxD').addEventListener('input',function(){opts.maxDensity=+this.value;opts.autoFit=false;$('maxDV').textContent=this.value+' /km\u00b2';updateEnds();paintSurface();bf();});
  function setFill(f){opts.fill=f;$('fillSmooth').classList.toggle('active',f==='smooth');$('fillSmooth').setAttribute('aria-pressed',f==='smooth');$('fillStepped').classList.toggle('active',f==='stepped');$('fillStepped').setAttribute('aria-pressed',f==='stepped');paintSurface();bf();}
  $('fillSmooth').addEventListener('click',function(){setFill('smooth');});
  $('fillStepped').addEventListener('click',function(){setFill('stepped');});
  $('hexSize').addEventListener('input',function(){opts.hexSize=+this.value;$('hexSizeV').textContent=this.value+' m';renderHex();bf();});
  $('hexCeil').addEventListener('input',function(){opts.hexCeil=+this.value;$('hexCeilV').textContent=this.value+' / hex';updateEnds();renderHex();bf();});
  function setMode(m){mode=m;var s=(m==='surface');$('modeSurface').classList.toggle('active',s);$('modeSurface').setAttribute('aria-pressed',s);$('modeHex').classList.toggle('active',!s);$('modeHex').setAttribute('aria-pressed',!s);
    $('grpSurface').hidden=!s;$('grpHex').hidden=s;$('noteSurface').hidden=!s;$('noteHex').hidden=s;updateEnds();render();}
  $('modeSurface').addEventListener('click',function(){setMode('surface');});
  $('modeHex').addEventListener('click',function(){setMode('hex');});
  $('toggle').addEventListener('click',function(){$('panel').classList.toggle('collapsed');});
  function toggleGreen(){if(!greenLayer)return;greenVisible=!greenVisible;var sw=$('greenSw');
    if(greenVisible){greenLayer.addTo(map);sw.classList.add('on');sw.setAttribute('aria-checked','true');}else{map.removeLayer(greenLayer);sw.classList.remove('on');sw.setAttribute('aria-checked','false');}}
  $('greenSw').addEventListener('click',toggleGreen);
  $('greenSw').addEventListener('keydown',function(e){if(e.key===' '||e.key==='Enter'){e.preventDefault();toggleGreen();}});

  // ---- init from embedded data ----
  if(BOUNDARY){clipRings=collectRings(BOUNDARY);clipLayer=L.geoJSON(BOUNDARY,{style:{color:'#c7d0da',weight:1.4,fill:false,dashArray:'4 4'}}).addTo(map);}
  if(GRASS&&GRASS.features&&GRASS.features.length){greenLayer=L.geoJSON(GRASS,{pane:'greenPane',style:{color:'#2f7a4d',weight:0.5,fillColor:'#41b06a',fillOpacity:0.4}}).addTo(map);}
  else{$('greenSec').style.display='none';}
  updateEnds();render();
  if(clipLayer){try{var bb=clipLayer.getBounds();if(bb.isValid())map.fitBounds(bb.pad(0.03));}catch(e){}}
  else if(points.length){var pb=L.latLngBounds(points);if(pb.isValid())map.fitBounds(pb.pad(0.05));}
})();
</script>
</body>
</html>
"""

## Write the map

In [9]:
from IPython.display import FileLink

html = (TEMPLATE
        .replace("%%POINTS%%",       json.dumps(POINTS))
        .replace("%%BOUNDARY%%",     json.dumps(BOUNDARY))
        .replace("%%GRASS%%",        json.dumps(GRASS))
        .replace("%%BASEMAP_URL%%",  json.dumps(BASEMAP_URL))
        .replace("%%BASEMAP_ATTR%%", json.dumps(BASEMAP_ATTR)))

with open(OUT_HTML, "w", encoding="utf-8") as f:
    f.write(html)

print(f"wrote {OUT_HTML}  ({len(html)/1024:.0f} KB, {len(POINTS)} crossings, "
      f"{len(GRASS['features'])} grass polygons)")
FileLink(OUT_HTML)

wrote boston_crosswalk_density.html  (4045 KB, 22451 crossings, 3683 grass polygons)


c:\Users\Sam\WalkSensePlace\boston_crosswalk_density.html